# KATS — Setup, Data Generation & Preprocessing

KATS Framework — Kinetic Attack Triage System


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
N = 15000

# Sector-conditional feature generation
sectors = ['banking', 'health', 'government', 'retail', 'ai_inference', 'batch_analytics']
sector_weights = [0.20, 0.15, 0.10, 0.20, 0.20, 0.15]
sector_labels = np.random.choice(sectors, size=N, p=sector_weights)

def generate_features(sector):
    if sector == 'banking':
        crit = np.random.randint(7, 11)
        rto = np.random.uniform(1, 30)
        reg = 1
    elif sector == 'health':
        crit = np.random.randint(6, 10)
        rto = np.random.uniform(5, 60)
        reg = 1
    elif sector == 'government':
        crit = np.random.randint(6, 10)
        rto = np.random.uniform(10, 60)
        reg = 1
    elif sector == 'ai_inference':
        crit = np.random.randint(5, 9)
        rto = np.random.uniform(5, 90)
        reg = 0
    elif sector == 'retail':
        crit = np.random.randint(3, 7)
        rto = np.random.uniform(30, 240)
        reg = 0
    else:  # batch_analytics
        crit = np.random.randint(1, 4)
        rto = np.random.uniform(120, 1440)
        reg = 0
    return crit, rto, reg

rows = []
for s in sector_labels:
    crit, rto, reg = generate_features(s)
    row = {
        'service_criticality': crit,
        'data_volume_gb': np.random.exponential(50),
        'rto_minutes': rto,
        'rpo_minutes': rto * np.random.uniform(0.3, 0.8),
        'dependency_count': np.random.poisson(3),
        'downstream_critical': np.random.binomial(1, 0.3 if crit > 6 else 0.05),
        'redundancy_level': np.random.randint(0, 4),
        'regulatory_flag': reg,
        'active_sessions': np.random.exponential(500),
        'bandwidth_required_mbps': np.random.exponential(100),
        'latency_sensitivity': np.random.binomial(1, 0.7 if crit > 6 else 0.2),
        'az_risk_score': np.random.beta(2, 5),
        'multi_region_deployed': np.random.binomial(1, 0.4),
        'service_sector': s,
        'migration_complexity': np.random.randint(1, 6),
    }
    rows.append(row)

df = pd.DataFrame(rows)

# Derive priority label using weighted formula
df['priority_score'] = (
    0.35 * df['service_criticality'] / 10 +
    0.20 * (1 - df['rto_minutes'] / df['rto_minutes'].max()) +
    0.15 * df['regulatory_flag'] +
    0.15 * df['downstream_critical'] +
    0.10 * df['az_risk_score'] +
    0.05 * (1 - df['redundancy_level'] / 3)
)

# Assign labels
df['priority_label'] = pd.cut(
    df['priority_score'],
    bins=[-np.inf, 0.35, 0.55, np.inf],
    labels=['Low', 'Medium', 'High']
)

print(df['priority_label'].value_counts())
print(df.shape)
df.to_csv('kats_syn_dataset.csv', index=False)
print("KATS-SYN saved!")

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (classification_report, f1_score, recall_score,
                              precision_score, cohen_kappa_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb
from statsmodels.stats.contingency_tables import mcnemar
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("All libraries loaded successfully.")

In [ ]:
import os

def list_all_files(base_path, max_files=30):
    count = 0
    for root, dirs, files in os.walk(base_path):
        for f in files:
            full = os.path.join(root, f)
            print(full)
            count += 1
            if count >= max_files:
                print("  ... (truncated)")
                return

print("=" * 60)
print("GOOGLE BORG FILES:")
list_all_files('/kaggle/input/google-2019-cluster-sample/')

print("\n" + "=" * 60)
print("BITBRAINS FILES:")
list_all_files('/kaggle/input/gwa-bitbrains/', max_files=10)

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import f1_score, recall_score, cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar
import lightgbm as lgb
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("✓ All imports successful")

In [ ]:
import os

print("ALL INPUT DATASETS:")
for entry in os.listdir('/kaggle/input'):
    full = os.path.join('/kaggle/input', entry)
    print(f"\n📁 /kaggle/input/{entry}/")
    for root, dirs, files in os.walk(full):
        for f in files:
            fpath = os.path.join(root, f)
            size  = os.path.getsize(fpath)
            print(f"   {fpath}  [{size/1024:.1f} KB]")

In [ ]:
import os, glob
import pandas as pd
import numpy as np

BITBRAINS_ROOT = '/kaggle/input/datasets/gauravdhamane/gwa-bitbrains/fastStorage/2013-8'

bb_files = sorted(glob.glob(os.path.join(BITBRAINS_ROOT, '*.csv')))
print(f"✓ Bitbrains files found: {len(bb_files)}")
print(f"  Sample: {bb_files[:3]}")

In [ ]:
# The Bitbrains CSVs have NO header row — space-separated, 8 columns
sample = pd.read_csv(bb_files[0], header=None, sep='\s+')
print("Shape:", sample.shape)
print(sample.head(3))

In [ ]:
# Official GWA-Bitbrains column order (no header in files):
BB_COLS = [
    'timestamp',
    'cpu_cores',
    'cpu_usage_mhz',
    'cpu_usage_pct',
    'memory_provisioned_kb',
    'memory_used_kb',
    'disk_read_throughput',
    'disk_write_throughput',
    'net_received_throughput',
    'net_transmitted_throughput',
]

records = []
for fpath in bb_files:
    try:
        df = pd.read_csv(fpath, header=None, sep='\s+', names=BB_COLS[:len(pd.read_csv(fpath, header=None, sep='\s+', nrows=1).columns)])
        df['vm_id'] = os.path.basename(fpath).replace('.csv','')
        records.append(df)
    except Exception as e:
        print(f"  ⚠ Skipped {fpath}: {e}")

df_bitbrains = pd.concat(records, ignore_index=True)
print(f"✓ Total rows: {df_bitbrains.shape[0]:,}  |  VMs: {df_bitbrains['vm_id'].nunique()}")
print(df_bitbrains.head(3))

In [ ]:
# Read one file to check actual column count
probe = pd.read_csv(bb_files[0], header=None, sep='\s+')
n_cols = probe.shape[1]
print(f"Detected {n_cols} columns per file")

# Assign column names up to what exists
col_names = BB_COLS[:n_cols]

frames = []
for fpath in bb_files:
    try:
        df = pd.read_csv(fpath, header=None, sep='\s+', names=col_names)
        if len(df) < 5:       # skip near-empty files (e.g. 753.csv = 1 KB)
            continue
        df['vm_id'] = os.path.basename(fpath).replace('.csv', '')
        frames.append(df)
    except Exception:
        pass

df_bitbrains = pd.concat(frames, ignore_index=True)
print(f"✓ Loaded {df_bitbrains['vm_id'].nunique()} VMs, {len(df_bitbrains):,} rows")
print("Columns:", df_bitbrains.columns.tolist())

In [ ]:
# Check if Borg dataset exists anywhere
borg_search = glob.glob('/kaggle/input/**/*borg*', recursive=True)[:5]
borg_search += glob.glob('/kaggle/input/**/*cluster*', recursive=True)[:5]
if borg_search:
    print("Found Borg-related files:", borg_search)
else:
    print("⚠ Google Borg dataset NOT found in /kaggle/input/")
    print("  → You need to add it via: Notebook → + Add Data → search 'google-borg-2019'")
    print("  → Recommended dataset: 'Google Cluster Data 2019' on Kaggle")

In [ ]:
BORG_PATH = '/kaggle/input/datasets/derrickmwiti/google-2019-cluster-sample/borg_traces_data.csv'

df_borg = pd.read_csv(BORG_PATH, low_memory=False)
df_borg.columns = [c.lower().strip().replace(' ', '_') for c in df_borg.columns]

print("Borg shape:", df_borg.shape)
print("Borg columns:", df_borg.columns.tolist())
print(df_borg.head(3))

In [ ]:
print("=== BORG QUALITY ===")
print(f"Null %:\n{df_borg.isnull().mean().mul(100).round(1).to_string()}")
print(f"\nEvent types: {df_borg['event'].value_counts().to_dict()}")
print(f"Failed distribution: {df_borg['failed'].value_counts().to_dict()}")

print("\n=== BITBRAINS QUALITY ===")
for col in ['cpu_usage_pct', 'memory_used_kb', 'cpu_usage_mhz']:
    if col in df_bitbrains.columns:
        print(f"{col}: nulls={df_bitbrains[col].isnull().sum()}, sample={df_bitbrains[col].dropna().iloc[0]}")

In [ ]:
import pandas as pd

ALIBABA_PATH = '/kaggle/input/datasets/derrickmwiti/cluster-trace-gpu-v2020'

# Check which files are usable
import os
for f in os.listdir(ALIBABA_PATH):
    fpath = os.path.join(ALIBABA_PATH, f)
    try:
        df = pd.read_csv(fpath, nrows=3)
        size = os.path.getsize(fpath) / (1024*1024)
        print(f"✓ {f}  [{size:.1f} MB]  shape_preview: {df.shape}  cols: {df.columns.tolist()[:5]}")
    except Exception as e:
        print(f"✗ {f}: {e}")

In [ ]:
import os, glob

AZURE_PATH = '/kaggle/input/datasets/mdhamidborkottulla/azure-vm-packing-2020'

# Walk ALL subdirectories
print("All files found (including subdirs):")
for root, dirs, files in os.walk(AZURE_PATH):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / (1024*1024)
        print(f"  {fpath}  [{size:.1f} MB]")

In [ ]:
# ============================================================
# CELL 1: Imports & Path Discovery
# ============================================================
import pandas as pd
import numpy as np
import glob
import os

BASE = '/kaggle/input/datasets/mdhamidborkottulla/azure-vm-packing-2020'
all_files = glob.glob(f'{BASE}/**/*.csv', recursive=True)
print(f"Total CSV files found: {len(all_files)}")
print("Sample paths:")
for f in all_files[:5]:
    print(" ", f)

In [ ]:
# ============================================================
# CELL 2: Load all benchmark CSVs into one DataFrame
# ============================================================
records = []
for fpath in all_files:
    fname = os.path.basename(fpath)
    # Parse metadata from filename
    parts = fpath.split('/')
    try:
        testsuite = [p for p in parts if 'testsuite' in fpath.split('testsuite')[1].split('/')[0:1]]
        testsuite_val = fpath.split('/testsuite/')[1].split('/')[0] if '/testsuite/' in fpath else 'unknown'
        testname_val  = fpath.split('testname')[1].split('vmlifespan')[0].strip() if 'testname' in fpath else 'unknown'
        lifespan_val  = 'long' if 'vmlifespanlong' in fpath else ('short' if 'vmlifespanshort' in fpath else 'unknown')
        region_val    = 'eastus' if 'vmregioneastus' in fpath else ('westus2' if 'vmregionwestus2' in fpath else 'unknown')
        sku_val       = 'D8sv5' if 'vmskuD8sv5' in fpath else ('B8ms' if 'vmskuB8ms' in fpath else 'unknown')
        
        df_tmp = pd.read_csv(fpath, low_memory=False)
        df_tmp['testsuite']   = testsuite_val
        df_tmp['testname']    = testname_val
        df_tmp['vm_lifespan'] = lifespan_val
        df_tmp['vm_region']   = region_val
        df_tmp['vm_sku']      = sku_val
        records.append(df_tmp)
    except Exception as e:
        print(f"  Skipped {fname}: {e}")

df_azure = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
print(f"\nCombined DataFrame shape: {df_azure.shape}")
print("Columns:", df_azure.columns.tolist()[:15])

In [ ]:
# ============================================================
# CELL 3: Explore structure — unique testsuites & testnames
# ============================================================
if not df_azure.empty:
    print("Unique test suites:", df_azure['testsuite'].unique())
    print("\nUnique VM SKUs:", df_azure['vm_sku'].unique())
    print("Unique regions:", df_azure['vm_region'].unique())
    print("Unique lifespans:", df_azure['vm_lifespan'].unique())
    print("\nValue column names:", [c for c in df_azure.columns if c not in
          ['testsuite','testname','vm_lifespan','vm_region','vm_sku']])

In [ ]:
import pandas as pd
import numpy as np
import glob, os, re

# ── Locate all 4 datasets ─────────────────────────────────────────────
BASE = '/kaggle/input'

def find_dataset(name_fragment):
    for entry in os.listdir(BASE):
        if name_fragment.lower() in entry.lower():
            return os.path.join(BASE, entry)
    return None

PATH_AZURE    = find_dataset('azure-vm-packing')
PATH_BORG     = find_dataset('google')
PATH_BITBRAINS = find_dataset('gwa-bitbrains') or find_dataset('bitbrains')
PATH_ALIBABA  = find_dataset('alibaba')

print("Azure VM Packing :", PATH_AZURE)
print("Google Borg      :", PATH_BORG)
print("BitBrains        :", PATH_BITBRAINS)
print("Alibaba GPU      :", PATH_ALIBABA)

# Quick file listing for each
for label, path in [("BORG", PATH_BORG), ("BITBRAINS", PATH_BITBRAINS)]:
    if path:
        files = glob.glob(f'{path}/**/*', recursive=True)
        files = [f for f in files if os.path.isfile(f)]
        print(f"\n{label} — {len(files)} files, sample:")
        for f in files[:5]: print(" ", f)
    else:
        print(f"\n{label} — NOT FOUND")

In [ ]:
# The Azure dataset uses Hive-style partitioned paths: test_suite=X/test_name=Y/...
# Extract metadata from the path itself, not from the filename

azure_files = glob.glob(f'{PATH_AZURE}/**/*.csv', recursive=True)
# Filter to only the benchmark files (inside vm-noise-data)
azure_files = [f for f in azure_files if 'vm-noise-data' in f]
print(f"Azure benchmark CSV files: {len(azure_files)}")

def parse_kv_path(fpath):
    """Extract key=value pairs from a Hive-partitioned path."""
    meta = {}
    for segment in fpath.split('/'):
        if '=' in segment:
            k, v = segment.split('=', 1)
            meta[k.strip()] = v.strip()
    return meta

records = []
for fpath in azure_files:
    try:
        meta = parse_kv_path(fpath)
        df_tmp = pd.read_csv(fpath, low_memory=False)
        for k, v in meta.items():
            df_tmp[k] = v
        records.append(df_tmp)
    except Exception as e:
        pass  # skip unreadable files silently

df_azure = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
print(f"Azure combined shape: {df_azure.shape}")
print("Columns:", df_azure.columns.tolist())

In [ ]:
# Google 2019 Cluster Trace — locate the right CSV files
borg_all = glob.glob(f'{PATH_BORG}/**/*.csv', recursive=True) if PATH_BORG else []
print(f"Borg CSV files found: {len(borg_all)}")
for f in borg_all[:10]:
    print(" ", f)

# Load instance_events or task_usage tables (most useful for scheduling research)
instance_files = [f for f in borg_all if 'instance_events' in f.lower()]
task_files      = [f for f in borg_all if 'task_usage' in f.lower() or 'instance_usage' in f.lower()]
job_files       = [f for f in borg_all if 'job_events' in f.lower()]

print(f"\nInstance event files : {len(instance_files)}")
print(f"Task/instance usage  : {len(task_files)}")
print(f"Job event files      : {len(job_files)}")

# Load whichever is available
target_files = instance_files or task_files or job_files or borg_all
if target_files:
    borg_raw = pd.read_csv(target_files[0], low_memory=False)
    print(f"\nLoaded: {target_files[0]}")
    print("Shape:", borg_raw.shape)
    print("Columns:", borg_raw.columns.tolist())
else:
    print("No Borg CSV files found — check dataset path")
    borg_raw = pd.DataFrame()

In [ ]:
bb_all = glob.glob(f'{PATH_BITBRAINS}/**/*', recursive=True) if PATH_BITBRAINS else []
bb_all = [f for f in bb_all if os.path.isfile(f)]
print(f"BitBrains total files: {len(bb_all)}")

# BitBrains files are typically semicolon-delimited .txt or .csv per VM
bb_data_files = [f for f in bb_all if f.endswith('.csv') or f.endswith('.txt')]
print(f"Data files (.csv/.txt): {len(bb_data_files)}")
for f in bb_data_files[:5]:
    print(" ", f)

# Peek at first file to understand delimiter and columns
if bb_data_files:
    with open(bb_data_files[0], 'r') as fh:
        head = [next(fh) for _ in range(3)]
    print("\nFirst 3 lines of", os.path.basename(bb_data_files[0]))
    for line in head:
        print(" ", line.rstrip())

In [ ]:
# BitBrains uses semicolon-separated format with header on line 1
# Columns: Timestamp [ms];CPU cores;CPU usage [%];CPU capacity provisioned [MHZ];
#           CPU usage [MHZ];Memory provisioned [KB];Memory usage [KB];...

bb_records = []
for fpath in bb_data_files[:200]:   # limit to 200 VMs for speed; remove cap for full run
    try:
        df_tmp = pd.read_csv(fpath, sep=';', low_memory=False)
        df_tmp.columns = [c.strip().lower().replace(' ', '_').replace('[', '').replace(']', '').replace('/', '_') for c in df_tmp.columns]
        df_tmp['vm_id'] = os.path.basename(fpath).replace('.csv', '').replace('.txt', '')
        bb_records.append(df_tmp)
    except Exception as e:
        pass

df_bitbrains = pd.concat(bb_records, ignore_index=True) if bb_records else pd.DataFrame()
print(f"BitBrains combined shape: {df_bitbrains.shape}")
print("Columns:", df_bitbrains.columns.tolist())
if not df_bitbrains.empty:
    print(df_bitbrains.head(3).to_string())

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"{'Dataset':<20} {'Rows':>10} {'Cols':>6}")
print("-" * 40)
for name, df in [("Azure VM Noise", df_azure), ("Google Borg", borg_raw), ("BitBrains", df_bitbrains)]:
    print(f"{name:<20} {len(df):>10,} {len(df.columns):>6}")

In [ ]:
import os, glob, pandas as pd, numpy as np

# Step 1: Find the actual root where Kaggle stores datasets in THIS notebook
def find_all_dirs(start='/kaggle/input', maxdepth=6):
    """Walk the full input tree and print everything."""
    for root, dirs, files in os.walk(start):
        depth = root.replace(start, '').count(os.sep)
        if depth > maxdepth:
            dirs[:] = []
            continue
        indent = '  ' * depth
        print(f"{indent}{root}/  [{len(files)} files]")
        if depth >= 3:
            dirs[:] = []  # don't go deeper than needed for discovery

find_all_dirs('/kaggle/input')

In [ ]:
import os, glob, pandas as pd, numpy as np

PATH_BORG      = '/kaggle/input/datasets/derrickmwiti/google-2019-cluster-sample'
PATH_ALIBABA   = '/kaggle/input/datasets/derrickmwiti/cluster-trace-gpu-v2020'
PATH_AZURE     = '/kaggle/input/datasets/mdhamidborkottulla/azure-vm-packing-2020'
PATH_BITBRAINS = '/kaggle/input/datasets/gauravdhamane/gwa-bitbrains'

for label, path in [("BORG", PATH_BORG), ("ALIBABA", PATH_ALIBABA),
                    ("AZURE", PATH_AZURE), ("BITBRAINS", PATH_BITBRAINS)]:
    all_files = glob.glob(f'{path}/**/*', recursive=True)
    all_files = [f for f in all_files if os.path.isfile(f)]
    print(f"\n{'='*55}")
    print(f"{label}: {len(all_files)} files total")
    for f in all_files[:6]:
        size = os.path.getsize(f) / 1024
        print(f"  [{size:6.0f} KB]  {f.replace(path, '')}")

In [ ]:
# Borg has 1 file — could be CSV, parquet, or zip
borg_files = glob.glob(f'{PATH_BORG}/**/*', recursive=True)
borg_files = [f for f in borg_files if os.path.isfile(f)]
print("Borg files:", borg_files)

borg_raw = pd.DataFrame()
for f in borg_files:
    ext = f.split('.')[-1].lower()
    try:
        if ext == 'csv':
            borg_raw = pd.read_csv(f, low_memory=False)
        elif ext == 'parquet':
            borg_raw = pd.read_parquet(f)
        elif ext in ['gz', 'zip']:
            borg_raw = pd.read_csv(f, compression='infer', low_memory=False)
        print(f"Loaded: {f}")
        print("Shape:", borg_raw.shape)
        print("Columns:", borg_raw.columns.tolist()[:10])
        break
    except Exception as e:
        print(f"Error reading {f}: {e}")

In [ ]:
# Azure uses Hive-partitioned paths: test_suite=X/test_name=Y/vm_lifespan=Z/...
azure_files = glob.glob(f'{PATH_AZURE}/**/*.csv', recursive=True)
azure_files = [f for f in azure_files if 'vm-noise-data' in f]
print(f"Azure benchmark CSVs: {len(azure_files)}")

def parse_kv_path(fpath):
    meta = {}
    for seg in fpath.split('/'):
        if '=' in seg:
            k, v = seg.split('=', 1)
            meta[k.strip()] = v.strip()
    return meta

records = []
for fpath in azure_files:
    try:
        meta = parse_kv_path(fpath)
        df_tmp = pd.read_csv(fpath, low_memory=False)
        for k, v in meta.items():
            df_tmp[k] = v
        records.append(df_tmp)
    except:
        pass

df_azure = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
print(f"Azure combined shape: {df_azure.shape}")
print("Columns:", df_azure.columns.tolist())

In [ ]:
alibaba_files = glob.glob(f'{PATH_ALIBABA}/**/*', recursive=True)
alibaba_files = [f for f in alibaba_files if os.path.isfile(f)]
print(f"Alibaba files: {len(alibaba_files)}")
for f in alibaba_files:
    print(" ", f.replace(PATH_ALIBABA, ''))

# Load first CSV found
for f in alibaba_files:
    if f.endswith('.csv'):
        df_alibaba = pd.read_csv(f, low_memory=False)
        print(f"\nLoaded: {os.path.basename(f)}")
        print("Shape:", df_alibaba.shape)
        print("Columns:", df_alibaba.columns.tolist())
        break

In [ ]:
# BitBrains showed 0 files — check if it's empty or has subdirs
print("BitBrains directory contents (all levels):")
for root, dirs, files in os.walk(PATH_BITBRAINS):
    print(f"  {root}/ — {len(files)} files, {len(dirs)} subdirs")
    for f in files[:3]:
        print(f"    {f}")

In [ ]:
# BitBrains files are in the deeper subdirectory
PATH_BITBRAINS_DATA = f'{PATH_BITBRAINS}/fastStorage/2013-8'
bb_files = glob.glob(f'{PATH_BITBRAINS_DATA}/*.csv')
print(f"BitBrains VM files: {len(bb_files)}")

# Peek at structure of first file
with open(bb_files[0], 'r') as fh:
    for i, line in enumerate(fh):
        print(line.rstrip())
        if i >= 3: break

In [ ]:
bb_records = []
for fpath in bb_files:
    try:
        df_tmp = pd.read_csv(fpath, sep=';', low_memory=False)
        # Normalize column names
        df_tmp.columns = [c.strip().lower()
                           .replace(' ', '_')
                           .replace('[', '').replace(']', '')
                           .replace('/', '_per_') for c in df_tmp.columns]
        df_tmp['vm_id'] = os.path.basename(fpath).replace('.csv', '')
        bb_records.append(df_tmp)
    except Exception as e:
        print(f"  Skipped {os.path.basename(fpath)}: {e}")

df_bitbrains = pd.concat(bb_records, ignore_index=True) if bb_records else pd.DataFrame()
print(f"BitBrains combined shape: {df_bitbrains.shape}")
print("Columns:", df_bitbrains.columns.tolist())
print(df_bitbrains.head(3).to_string())

In [ ]:
borg_raw.columns = [c.strip().lower().replace(' ', '_') for c in borg_raw.columns]

# Drop unnamed index column if present
if 'unnamed:_0' in borg_raw.columns:
    borg_raw = borg_raw.drop(columns=['unnamed:_0'])

# Key columns for scheduling research
key_cols = ['time', 'instance_events_type', 'collection_id',
            'scheduling_class', 'priority', 'machine_id',
            'collection_type', 'alloc_collection_id', 'instance_index']

key_cols = [c for c in key_cols if c in borg_raw.columns]
df_borg = borg_raw[key_cols].copy()

# Check for CPU/memory resource columns
resource_cols = [c for c in borg_raw.columns if any(k in c for k in
                 ['cpu', 'mem', 'resource', 'request', 'limit', 'usage'])]
print("Resource-related columns:", resource_cols)

if resource_cols:
    df_borg = borg_raw[key_cols + resource_cols].copy()

print(f"\nBorg normalized shape: {df_borg.shape}")
print("Columns:", df_borg.columns.tolist())
print(df_borg.head(3).to_string())

In [ ]:
alibaba_tables = {}
for f in alibaba_files:
    name = os.path.basename(f).replace('.csv', '')
    try:
        alibaba_tables[name] = pd.read_csv(f, low_memory=False)
        print(f"  {name}: {alibaba_tables[name].shape}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

# Most useful for scheduling: instance + task + machine_spec
df_alibaba_inst = alibaba_tables.get('pai_instance_table', pd.DataFrame())
df_alibaba_task = alibaba_tables.get('pai_task_table', pd.DataFrame())
df_alibaba_spec = alibaba_tables.get('pai_machine_spec', pd.DataFrame())

print("\nInstance table columns:", df_alibaba_inst.columns.tolist())
print("Task table columns:", df_alibaba_task.columns.tolist())

In [ ]:
print("=" * 60)
print("ALL DATASETS LOADED SUCCESSFULLY")
print("=" * 60)
datasets = {
    "Google Borg":    df_borg,
    "Azure VM Noise": df_azure,
    "BitBrains":      df_bitbrains,
    "Alibaba Inst":   df_alibaba_inst,
    "Alibaba Task":   df_alibaba_task,
    "Alibaba Spec":   df_alibaba_spec,
}
for name, df in datasets.items():
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"  {name:<20} {df.shape[0]:>10,} rows × {df.shape[1]:>3} cols   [{mb:.1f} MB]")

In [ ]:
import ast

def extract_resource(series, key):
    """Parse dict-like string column and extract a numeric key."""
    def safe_get(val):
        try:
            if pd.isna(val): return np.nan
            d = ast.literal_eval(str(val).replace("'", '"').replace('None', 'null'))
            return d.get(key, np.nan)
        except:
            return np.nan
    return series.apply(safe_get)

# Extract CPU and memory from the dict-format columns
df_borg['req_cpu']  = extract_resource(df_borg['resource_request'], 'cpus')
df_borg['req_mem']  = extract_resource(df_borg['resource_request'], 'memory')
df_borg['avg_cpu']  = extract_resource(df_borg['average_usage'],    'cpus')
df_borg['avg_mem']  = extract_resource(df_borg['average_usage'],    'memory')
df_borg['max_cpu']  = extract_resource(df_borg['maximum_usage'],    'cpus')
df_borg['max_mem']  = extract_resource(df_borg['maximum_usage'],    'memory')

print("Borg with extracted resources:")
print(df_borg[['priority','req_cpu','req_mem','avg_cpu','avg_mem','max_cpu','max_mem']].describe().round(4))

In [ ]:
# Compute utilization ratios (0–100%)
df_bitbrains['cpu_util_pct']    = df_bitbrains['cpu_usage_%']
df_bitbrains['mem_util_pct']    = (df_bitbrains['memory_usage_kb'] /
                                    df_bitbrains['memory_capacity_provisioned_kb'] * 100).clip(0, 100)

# Convert timestamp to datetime
df_bitbrains['timestamp'] = pd.to_datetime(df_bitbrains['timestamp_ms'], unit='s')

print("BitBrains utilization stats:")
print(df_bitbrains[['cpu_util_pct', 'mem_util_pct',
                     'disk_read_throughput_kb_per_s',
                     'disk_write_throughput_kb_per_s']].describe().round(2))

In [ ]:
# Azure: compute per-SKU, per-test mean/std/CoV for noise analysis
azure_stats = (df_azure.groupby(['test_suite', 'test_name', 'vm_sku', 'vm_region', 'unit'])['value']
               .agg(mean='mean', std='std', count='count')
               .reset_index())
azure_stats['cv_pct'] = (azure_stats['std'] / azure_stats['mean'] * 100).round(2)  # Coefficient of Variation

print(f"Azure stats shape: {azure_stats.shape}")
print(azure_stats[azure_stats['test_suite'] == 'Sysbench'][
    ['test_name','vm_sku','vm_region','mean','std','cv_pct']].head(10).to_string())

In [ ]:
def downcast_df(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

df_borg      = downcast_df(df_borg)
df_bitbrains = downcast_df(df_bitbrains)
df_azure     = downcast_df(df_azure)

# Print memory savings
print("Memory after downcasting:")
for name, df in [("Borg", df_borg), ("BitBrains", df_bitbrains), ("Azure", df_azure)]:
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"  {name:<12}: {mb:.1f} MB")

In [ ]:
import os
os.makedirs('/kaggle/working/clean', exist_ok=True)

df_borg.to_parquet('/kaggle/working/clean/borg.parquet', index=False)
df_bitbrains.to_parquet('/kaggle/working/clean/bitbrains.parquet', index=False)
df_azure.to_parquet('/kaggle/working/clean/azure_noise.parquet', index=False)
df_alibaba_inst.to_parquet('/kaggle/working/clean/alibaba_instance.parquet', index=False)
df_alibaba_task.to_parquet('/kaggle/working/clean/alibaba_task.parquet', index=False)

print("✅ All cleaned datasets saved to /kaggle/working/clean/")
for f in os.listdir('/kaggle/working/clean'):
    size = os.path.getsize(f'/kaggle/working/clean/{f}') / 1e6
    print(f"  {f:<35} {size:.1f} MB")

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

N = 15000

# Sector definitions with conditional priors
sectors = ['banking', 'health', 'government', 'retail', 'ai_inference', 'batch_analytics']
sector_weights = [0.20, 0.15, 0.10, 0.20, 0.20, 0.15]

sector_priors = {
    'banking':       dict(crit=(7,10), rto=(1,30),    regulatory=1, latency=1, redundancy=(0,2)),
    'health':        dict(crit=(6,10), rto=(5,60),    regulatory=1, latency=1, redundancy=(0,2)),
    'government':    dict(crit=(5,9),  rto=(10,120),  regulatory=1, latency=0, redundancy=(0,3)),
    'retail':        dict(crit=(3,7),  rto=(15,240),  regulatory=0, latency=1, redundancy=(1,3)),
    'ai_inference':  dict(crit=(5,9),  rto=(1,60),    regulatory=0, latency=1, redundancy=(0,2)),
    'batch_analytics':dict(crit=(1,4), rto=(120,1440),regulatory=0, latency=0, redundancy=(1,3)),
}

rows = []
for _ in range(N):
    sector = np.random.choice(sectors, p=sector_weights)
    p = sector_priors[sector]

    criticality       = np.random.randint(p['crit'][0], p['crit'][1]+1)
    rto_minutes       = np.random.uniform(p['rto'][0], p['rto'][1])
    rpo_minutes       = rto_minutes * np.random.uniform(0.3, 0.8)
    data_volume_gb    = np.random.exponential(50) + 1
    dependency_count  = np.random.poisson(3)
    downstream_critical = int(np.random.random() < (0.6 if criticality >= 7 else 0.2))
    redundancy_level  = np.random.randint(p['redundancy'][0], p['redundancy'][1]+1)
    regulatory_flag   = p['regulatory']
    active_sessions   = int(np.random.exponential(5000) * (criticality / 10))
    bandwidth_mbps    = data_volume_gb * 8 / (rto_minutes * 60 / 1000) * np.random.uniform(0.8, 1.2)
    latency_sensitive = p['latency']
    az_risk_score     = np.random.beta(2, 5) + (0.3 if sector in ['banking','ai_inference'] else 0.0)
    az_risk_score     = min(az_risk_score, 1.0)
    multiregion       = int(redundancy_level >= 2)
    migration_complexity = np.random.randint(1, 6)

    # Weighted priority label formula
    score = (0.30 * criticality / 10 +
             0.20 * (1 - min(rto_minutes, 1440) / 1440) +
             0.15 * regulatory_flag +
             0.15 * downstream_critical +
             0.10 * az_risk_score +
             0.05 * latency_sensitive +
             0.05 * (1 - redundancy_level / 3))

    if score >= 0.55:
        label = 'High'
    elif score >= 0.30:
        label = 'Medium'
    else:
        label = 'Low'

    rows.append({
        'service_criticality': criticality,
        'data_volume_gb': round(data_volume_gb, 2),
        'rto_minutes': round(rto_minutes, 1),
        'rpo_minutes': round(rpo_minutes, 1),
        'dependency_count': dependency_count,
        'downstream_critical': downstream_critical,
        'redundancy_level': redundancy_level,
        'regulatory_flag': regulatory_flag,
        'active_sessions': active_sessions,
        'bandwidth_required_mbps': round(bandwidth_mbps, 2),
        'latency_sensitivity': latency_sensitive,
        'az_risk_score': round(az_risk_score, 4),
        'multiregion_deployed': multiregion,
        'service_sector': sector,
        'migration_complexity': migration_complexity,
        'priority_label': label,
        'priority_score': round(score, 4)
    })

df_kats_syn = pd.DataFrame(rows)

# Add 5% label noise (realistic uncertainty)
noise_idx = df_kats_syn.sample(frac=0.05, random_state=42).index
labels = ['High', 'Medium', 'Low']
df_kats_syn.loc[noise_idx, 'priority_label'] = np.random.choice(labels, size=len(noise_idx))

print(f"KATS-SYN shape: {df_kats_syn.shape}")
print("\nLabel distribution:")
print(df_kats_syn['priority_label'].value_counts())
print("\nSector distribution:")
print(df_kats_syn['service_sector'].value_counts())
print("\nSample:")
print(df_kats_syn.head(3).to_string())

In [ ]:
df_borg_kats = pd.DataFrame()

# Borg priority (0–450) → service_criticality (1–10)
df_borg_kats['service_criticality'] = (df_borg['priority'] / 450 * 9 + 1).clip(1, 10).round().astype(int)

# scheduling_class (0=batch → 3=latency-sensitive) → latency_sensitivity
df_borg_kats['latency_sensitivity'] = (df_borg['scheduling_class'] >= 2).astype(int)

# CPU & memory request → migration_complexity proxy
df_borg_kats['bandwidth_required_mbps'] = (df_borg['req_cpu'].fillna(0) * 1000 +
                                            df_borg['req_mem'].fillna(0) * 512).round(2)
df_borg_kats['migration_complexity']    = pd.cut(df_borg_kats['bandwidth_required_mbps'],
                                                  bins=[-1,50,200,500,1000,99999],
                                                  labels=[1,2,3,4,5]).astype(float).fillna(3).astype(int)

# avg_cpu → active_sessions proxy
df_borg_kats['active_sessions'] = (df_borg['avg_cpu'].fillna(0) * 10000).astype(int)

# Memory usage → data_volume_gb proxy (Borg memory is normalized 0–1 fraction of machine)
df_borg_kats['data_volume_gb']  = (df_borg['avg_mem'].fillna(0) * 256).round(2)  # assume 256GB machines

# RTO: high-priority jobs have tight RTO
df_borg_kats['rto_minutes']     = ((1 - df_borg['priority'] / 450) * 1440 + 1).round(1)
df_borg_kats['rpo_minutes']     = (df_borg_kats['rto_minutes'] * 0.5).round(1)

# instance_events_type 2=submit, 3=schedule, 4=evict, 5=fail, 6=finish
df_borg_kats['downstream_critical'] = (df_borg['instance_events_type'].isin([4, 5])).astype(int)
df_borg_kats['dependency_count']    = df_borg['alloc_collection_id'].apply(lambda x: 1 if x != 0 else 0)
df_borg_kats['redundancy_level']    = np.random.randint(0, 3, size=len(df_borg))
df_borg_kats['regulatory_flag']     = 0   # Borg has no regulatory info
df_borg_kats['az_risk_score']       = np.random.beta(2, 5, size=len(df_borg)).round(4)
df_borg_kats['multiregion_deployed']= (df_borg_kats['redundancy_level'] >= 2).astype(int)
df_borg_kats['service_sector']      = 'google_borg'

# Priority label: top-30% = High, middle-40% = Medium, bottom-30% = Low
q70 = df_borg_kats['service_criticality'].quantile(0.70)
q30 = df_borg_kats['service_criticality'].quantile(0.30)
df_borg_kats['priority_label'] = np.where(df_borg_kats['service_criticality'] >= q70, 'High',
                                  np.where(df_borg_kats['service_criticality'] <= q30, 'Low', 'Medium'))

# Keep only KATS schema columns
kats_cols = ['service_criticality','data_volume_gb','rto_minutes','rpo_minutes',
             'dependency_count','downstream_critical','redundancy_level','regulatory_flag',
             'active_sessions','bandwidth_required_mbps','latency_sensitivity',
             'az_risk_score','multiregion_deployed','service_sector',
             'migration_complexity','priority_label']
df_borg_kats = df_borg_kats[kats_cols]

print(f"Borg KATS shape: {df_borg_kats.shape}")
print(df_borg_kats['priority_label'].value_counts())

In [ ]:
# Aggregate per-VM: compute summary stats across time series
bb_agg = df_bitbrains.groupby('vm_id').agg(
    cpu_cores         = ('cpu_cores', 'first'),
    cpu_usage_pct_mean= ('cpu_util_pct', 'mean'),
    cpu_usage_pct_max = ('cpu_util_pct', 'max'),
    mem_prov_kb       = ('memory_capacity_provisioned_kb', 'first'),
    mem_usage_pct_mean= ('mem_util_pct', 'mean'),
    disk_write_max    = ('disk_write_throughput_kb_per_s', 'max'),
    net_rx_mean       = ('network_received_throughput_kb_per_s', 'mean'),
    net_tx_mean       = ('network_transmitted_throughput_kb_per_s', 'mean'),
).reset_index()

print(f"BitBrains per-VM aggregated: {bb_agg.shape}")

df_bb_kats = pd.DataFrame()
# All BitBrains VMs are from financial sector (banks, insurers) → regulatory_flag = 1
df_bb_kats['regulatory_flag']      = 1
df_bb_kats['service_sector']       = 'financial_bitbrains'

# CPU cores → migration_complexity
df_bb_kats['migration_complexity'] = pd.cut(bb_agg['cpu_cores'],
                                             bins=[0,1,2,4,8,999],
                                             labels=[1,2,3,4,5]).astype(float).fillna(2).astype(int)

# Memory provisioned → data_volume_gb
df_bb_kats['data_volume_gb']       = (bb_agg['mem_prov_kb'] / 1024 / 1024).round(2)  # KB → GB

# Network throughput → bandwidth_required_mbps
df_bb_kats['bandwidth_required_mbps'] = ((bb_agg['net_rx_mean'] + bb_agg['net_tx_mean']) / 1024 * 8).round(2)

# CPU usage → active_sessions proxy
df_bb_kats['active_sessions']      = (bb_agg['cpu_usage_pct_mean'] * 100).astype(int)

# High disk write → high RPO sensitivity (financial transaction logs)
df_bb_kats['rpo_minutes']          = (1 / (bb_agg['disk_write_max'].clip(1) / 1024 + 0.01)).clip(1, 240).round(1)
df_bb_kats['rto_minutes']          = (df_bb_kats['rpo_minutes'] * 2).clip(2, 480).round(1)

# CPU usage mean → service_criticality (financial workloads: high utilization = critical)
df_bb_kats['service_criticality']  = (bb_agg['cpu_usage_pct_mean'].clip(0,100) / 100 * 7 + 3).clip(1,10).round().astype(int)
df_bb_kats['latency_sensitivity']  = 1   # All financial = latency sensitive
df_bb_kats['downstream_critical']  = (bb_agg['cpu_usage_pct_max'] > 70).astype(int)
df_bb_kats['dependency_count']     = np.random.poisson(2, size=len(bb_agg))
df_bb_kats['redundancy_level']     = np.random.randint(0, 3, size=len(bb_agg))
df_bb_kats['az_risk_score']        = np.random.beta(3, 4, size=len(bb_agg)).round(4)
df_bb_kats['multiregion_deployed'] = (df_bb_kats['redundancy_level'] >= 2).astype(int)

# Priority label using same weighted formula as KATS-SYN
score = (0.30 * df_bb_kats['service_criticality'] / 10 +
         0.20 * (1 - df_bb_kats['rto_minutes'].clip(0,1440) / 1440) +
         0.15 * df_bb_kats['regulatory_flag'] +
         0.15 * df_bb_kats['downstream_critical'] +
         0.10 * df_bb_kats['az_risk_score'] +
         0.05 * df_bb_kats['latency_sensitivity'] +
         0.05 * (1 - df_bb_kats['redundancy_level'] / 3))

df_bb_kats['priority_label'] = np.where(score >= 0.55, 'High',
                                np.where(score >= 0.30, 'Medium', 'Low'))
df_bb_kats['vm_id'] = bb_agg['vm_id'].values
df_bb_kats = df_bb_kats[kats_cols]

print(f"BitBrains KATS shape: {df_bb_kats.shape}")
print(df_bb_kats['priority_label'].value_counts())

In [ ]:
# Use task table + machine spec — AI inference workloads
df_alib = df_alibaba_task.merge(df_alibaba_spec.rename(columns={'machine':'worker_name'}),
                                 on='worker_name', how='left') if 'worker_name' in df_alibaba_task.columns else df_alibaba_task.copy()

df_al_kats = pd.DataFrame()
df_al_kats['service_sector']       = 'ai_inference_alibaba'
df_al_kats['regulatory_flag']      = 0

# plan_cpu → migration_complexity
if 'plan_cpu' in df_alib.columns:
    df_al_kats['migration_complexity'] = pd.cut(df_alib['plan_cpu'].fillna(1),
                                                  bins=[0,2,4,8,16,9999],
                                                  labels=[1,2,3,4,5]).astype(float).fillna(2).astype(int)
    df_al_kats['bandwidth_required_mbps'] = (df_alib['plan_cpu'].fillna(0) * 100).round(2)
    df_al_kats['data_volume_gb']          = (df_alib['plan_mem'].fillna(0) / 1024).round(2)
else:
    df_al_kats['migration_complexity']    = 3
    df_al_kats['bandwidth_required_mbps'] = 100.0
    df_al_kats['data_volume_gb']          = 10.0

# GPU jobs with plan_gpu > 0 → high criticality AI inference
if 'plan_gpu' in df_alib.columns:
    df_al_kats['service_criticality']  = (df_alib['plan_gpu'].fillna(0).clip(0,8) / 8 * 6 + 4).clip(1,10).round().astype(int)
    df_al_kats['latency_sensitivity']  = (df_alib['plan_gpu'].fillna(0) > 0).astype(int)
else:
    df_al_kats['service_criticality']  = 6
    df_al_kats['latency_sensitivity']  = 1

# Duration → RTO (short tasks = tight RTO)
if 'start_time' in df_alib.columns and 'end_time' in df_alib.columns:
    duration = (pd.to_numeric(df_alib['end_time'], errors='coerce') -
                pd.to_numeric(df_alib['start_time'], errors='coerce')).fillna(3600)
    df_al_kats['rto_minutes'] = (duration / 60).clip(1, 1440).round(1)
else:
    df_al_kats['rto_minutes'] = 60.0

df_al_kats['rpo_minutes']           = (df_al_kats['rto_minutes'] * 0.5).round(1)
df_al_kats['active_sessions']       = (df_al_kats['service_criticality'] * 500).astype(int)
df_al_kats['downstream_critical']   = (df_al_kats['service_criticality'] >= 8).astype(int)
df_al_kats['dependency_count']      = np.random.poisson(2, size=len(df_alib))
df_al_kats['redundancy_level']      = np.random.randint(0, 3, size=len(df_alib))
df_al_kats['az_risk_score']         = np.random.beta(3, 3, size=len(df_alib)).round(4)  # Higher for Gulf AI
df_al_kats['multiregion_deployed']  = (df_al_kats['redundancy_level'] >= 2).astype(int)

score = (0.30 * df_al_kats['service_criticality'] / 10 +
         0.20 * (1 - df_al_kats['rto_minutes'].clip(0,1440) / 1440) +
         0.15 * df_al_kats['regulatory_flag'] +
         0.15 * df_al_kats['downstream_critical'] +
         0.10 * df_al_kats['az_risk_score'] +
         0.05 * df_al_kats['latency_sensitivity'] +
         0.05 * (1 - df_al_kats['redundancy_level'] / 3))

df_al_kats['priority_label']        = np.where(score >= 0.55, 'High',
                                       np.where(score >= 0.30, 'Medium', 'Low'))
df_al_kats = df_al_kats[kats_cols].reset_index(drop=True)

print(f"Alibaba KATS shape: {df_al_kats.shape}")
print(df_al_kats['priority_label'].value_counts())

In [ ]:
import os
os.makedirs('/kaggle/working/kats_data', exist_ok=True)

df_kats_syn.to_parquet('/kaggle/working/kats_data/kats_syn.parquet',      index=False)
df_borg_kats.to_parquet('/kaggle/working/kats_data/kats_borg.parquet',    index=False)
df_bb_kats.to_parquet('/kaggle/working/kats_data/kats_bitbrains.parquet', index=False)
df_al_kats.to_parquet('/kaggle/working/kats_data/kats_alibaba.parquet',   index=False)

print("✅ All 4 KATS-schema datasets saved!")
print("\nFINAL DATASET SUMMARY:")
print(f"{'Dataset':<25} {'Rows':>10} {'High':>8} {'Medium':>8} {'Low':>8}")
print("-" * 62)
for name, df in [("KATS-SYN (synthetic)", df_kats_syn),
                  ("Google Borg (mapped)", df_borg_kats),
                  ("BitBrains (mapped)",   df_bb_kats),
                  ("Alibaba GPU (mapped)", df_al_kats)]:
    vc = df['priority_label'].value_counts()
    print(f"{name:<25} {len(df):>10,} {vc.get('High',0):>8,} {vc.get('Medium',0):>8,} {vc.get('Low',0):>8,}")

In [ ]:
# Recompute score for Borg using same formula then apply PERCENTILE thresholds
borg_score = (0.30 * df_borg_kats['service_criticality'] / 10 +
              0.20 * (1 - df_borg_kats['rto_minutes'].clip(0,1440) / 1440) +
              0.15 * df_borg_kats['regulatory_flag'] +
              0.15 * df_borg_kats['downstream_critical'] +
              0.10 * df_borg_kats['az_risk_score'] +
              0.05 * df_borg_kats['latency_sensitivity'] +
              0.05 * (1 - df_borg_kats['redundancy_level'] / 3))

q70_borg = borg_score.quantile(0.70)
q30_borg = borg_score.quantile(0.30)
df_borg_kats['priority_label'] = np.where(borg_score >= q70_borg, 'High',
                                  np.where(borg_score >= q30_borg, 'Medium', 'Low'))
df_borg_kats['priority_score'] = borg_score.round(4)

print("Borg label distribution (percentile-based):")
print(df_borg_kats['priority_label'].value_counts())
print(f"Thresholds: High >= {q70_borg:.4f}, Low < {q30_borg:.4f}")

In [ ]:
bb_score = (0.30 * df_bb_kats['service_criticality'] / 10 +
            0.20 * (1 - df_bb_kats['rto_minutes'].clip(0,1440) / 1440) +
            0.15 * df_bb_kats['regulatory_flag'] +
            0.15 * df_bb_kats['downstream_critical'] +
            0.10 * df_bb_kats['az_risk_score'] +
            0.05 * df_bb_kats['latency_sensitivity'] +
            0.05 * (1 - df_bb_kats['redundancy_level'] / 3))

q70_bb = bb_score.quantile(0.70)
q30_bb = bb_score.quantile(0.30)
df_bb_kats['priority_label'] = np.where(bb_score >= q70_bb, 'High',
                                np.where(bb_score >= q30_bb, 'Medium', 'Low'))
df_bb_kats['priority_score'] = bb_score.round(4)

print("BitBrains label distribution (percentile-based):")
print(df_bb_kats['priority_label'].value_counts())
print(f"Thresholds: High >= {q70_bb:.4f}, Low < {q30_bb:.4f}")

In [ ]:
al_score = (0.30 * df_al_kats['service_criticality'] / 10 +
            0.20 * (1 - df_al_kats['rto_minutes'].clip(0,1440) / 1440) +
            0.15 * df_al_kats['regulatory_flag'] +
            0.15 * df_al_kats['downstream_critical'] +
            0.10 * df_al_kats['az_risk_score'] +
            0.05 * df_al_kats['latency_sensitivity'] +
            0.05 * (1 - df_al_kats['redundancy_level'] / 3))

q70_al = al_score.quantile(0.70)
q30_al = al_score.quantile(0.30)
df_al_kats['priority_label'] = np.where(al_score >= q70_al, 'High',
                                np.where(al_score >= q30_al, 'Medium', 'Low'))
df_al_kats['priority_score'] = al_score.round(4)

# Sample 50k rows — sufficient for cross-dataset validation, avoids memory issues
df_al_kats_sample = df_al_kats.sample(n=50000, random_state=42).reset_index(drop=True)

print("Alibaba label distribution (full):")
print(df_al_kats['priority_label'].value_counts())
print(f"\nAliaba sample (50k) distribution:")
print(df_al_kats_sample['priority_label'].value_counts())

In [ ]:
df_borg_kats.to_parquet('/kaggle/working/kats_data/kats_borg.parquet',         index=False)
df_bb_kats.to_parquet('/kaggle/working/kats_data/kats_bitbrains.parquet',       index=False)
df_al_kats_sample.to_parquet('/kaggle/working/kats_data/kats_alibaba.parquet',  index=False)
# KATS-SYN is already saved correctly

print("✅ All fixed datasets saved!")
print("\n" + "="*65)
print(f"{'Dataset':<26} {'Rows':>8} {'High%':>7} {'Med%':>7} {'Low%':>7}")
print("="*65)
for name, df in [("KATS-SYN (synthetic)",   df_kats_syn),
                  ("Google Borg (mapped)",    df_borg_kats),
                  ("BitBrains (mapped)",      df_bb_kats),
                  ("Alibaba GPU (50k sample)",df_al_kats_sample)]:
    vc  = df['priority_label'].value_counts(normalize=True) * 100
    print(f"{name:<26} {len(df):>8,}  {vc.get('High',0):>5.1f}%  {vc.get('Medium',0):>5.1f}%  {vc.get('Low',0):>5.1f}%")

print("\n✅ Ready for Week 5-6: Baseline Models + KATS-Ensemble training")

In [ ]:
# Check variance of each feature for BitBrains and Alibaba
print("=== BITBRAINS FEATURE VARIANCE ===")
bb_numeric = df_bb_kats.select_dtypes(include='number')
print(bb_numeric.var().round(6).sort_values())

print("\n=== ALIBABA FEATURE VARIANCE ===")
al_numeric = df_al_kats.select_dtypes(include='number')
print(al_numeric.var().round(6).sort_values())

print("\n=== BITBRAINS feature distributions ===")
print(df_bb_kats[['service_criticality','rto_minutes','downstream_critical',
                   'az_risk_score','regulatory_flag','latency_sensitivity',
                   'redundancy_level']].describe().round(3))

print("\n=== ALIBABA feature distributions ===")
print(df_al_kats[['service_criticality','rto_minutes','downstream_critical',
                   'az_risk_score','regulatory_flag','latency_sensitivity',
                   'redundancy_level']].describe().round(3))

In [ ]:
# Re-aggregate BitBrains with richer features
bb_agg2 = df_bitbrains.groupby('vm_id').agg(
    cpu_cores          = ('cpu_cores', 'first'),
    cpu_pct_mean       = ('cpu_util_pct', 'mean'),
    cpu_pct_max        = ('cpu_util_pct', 'max'),
    cpu_pct_std        = ('cpu_util_pct', 'std'),
    cpu_mhz_mean       = ('cpu_usage_mhz', 'mean'),
    cpu_mhz_max        = ('cpu_usage_mhz', 'max'),
    mem_prov_kb        = ('memory_capacity_provisioned_kb', 'first'),
    mem_usage_kb_mean  = ('memory_usage_kb', 'mean'),
    mem_usage_kb_max   = ('memory_usage_kb', 'max'),
    disk_read_mean     = ('disk_read_throughput_kb_per_s', 'mean'),
    disk_write_mean    = ('disk_write_throughput_kb_per_s', 'mean'),
    disk_write_max     = ('disk_write_throughput_kb_per_s', 'max'),
    net_rx_mean        = ('network_received_throughput_kb_per_s', 'mean'),
    net_tx_mean        = ('network_transmitted_throughput_kb_per_s', 'mean'),
    observations       = ('timestamp_ms', 'count'),
).reset_index()

print("BitBrains aggregated shape:", bb_agg2.shape)
print(bb_agg2.describe().round(2))

In [ ]:
df_bb2 = pd.DataFrame()

# service_criticality: blend of CPU load, memory utilization, disk write intensity
# All 3 vary per VM — this ensures variance
mem_util_ratio = (bb_agg2['mem_usage_kb_mean'] / bb_agg2['mem_prov_kb'].clip(1)).clip(0, 1)
cpu_norm       = (bb_agg2['cpu_pct_mean'] / 100).clip(0, 1)
disk_norm      = (bb_agg2['disk_write_mean'] / bb_agg2['disk_write_mean'].max().clip(1)).clip(0, 1)

df_bb2['service_criticality'] = ((cpu_norm * 4 + mem_util_ratio * 3 + disk_norm * 3) * 10 / 10).clip(0,1)
df_bb2['service_criticality'] = (df_bb2['service_criticality'] * 9 + 1).round().astype(int).clip(1, 10)

# rto_minutes: inverse of disk write rate (high write = low RTO tolerance)
disk_write_capped = bb_agg2['disk_write_mean'].clip(0.01, None)
df_bb2['rto_minutes'] = (1000 / disk_write_capped).clip(1, 1440).round(1)

df_bb2['rpo_minutes'] = (df_bb2['rto_minutes'] * 0.5).round(1)

# bandwidth: actual network I/O
df_bb2['bandwidth_required_mbps'] = ((bb_agg2['net_rx_mean'] + bb_agg2['net_tx_mean']) / 1024 * 8).clip(0.01).round(3)

# data_volume_gb: memory footprint
df_bb2['data_volume_gb'] = (bb_agg2['mem_usage_kb_mean'] / 1024 / 1024).round(3)

# active_sessions: proxy from CPU usage * cores
df_bb2['active_sessions'] = (bb_agg2['cpu_pct_mean'] * bb_agg2['cpu_cores']).astype(int)

# downstream_critical: high CPU variability = critical real-time service
df_bb2['downstream_critical'] = (bb_agg2['cpu_pct_std'].fillna(0) > bb_agg2['cpu_pct_mean'] * 0.5).astype(int)

# migration_complexity: based on CPU cores
df_bb2['migration_complexity'] = pd.cut(bb_agg2['cpu_cores'],
                                          bins=[0,1,2,4,8,999],
                                          labels=[1,2,3,4,5]).astype(float).fillna(2).astype(int)

# Fixed fields
df_bb2['regulatory_flag']      = 1   # All BitBrains = financial sector
df_bb2['latency_sensitivity']  = 1   # All financial = latency sensitive
df_bb2['redundancy_level']     = np.random.randint(0, 4, size=len(bb_agg2))
df_bb2['az_risk_score']        = (cpu_norm * 0.5 + mem_util_ratio * 0.3 + np.random.uniform(0, 0.2, len(bb_agg2))).clip(0,1).round(4)
df_bb2['multiregion_deployed'] = (df_bb2['redundancy_level'] >= 2).astype(int)
df_bb2['dependency_count']     = np.random.poisson(3, size=len(bb_agg2))
df_bb2['service_sector']       = 'financial_bitbrains'

# Compute score with real variance
bb2_score = (0.30 * df_bb2['service_criticality'] / 10 +
             0.20 * (1 - df_bb2['rto_minutes'].clip(0,1440) / 1440) +
             0.15 * df_bb2['regulatory_flag'] +
             0.15 * df_bb2['downstream_critical'] +
             0.10 * df_bb2['az_risk_score'] +
             0.05 * df_bb2['latency_sensitivity'] +
             0.05 * (1 - df_bb2['redundancy_level'] / 3))

print(f"\nBitBrains score stats:\n{bb2_score.describe().round(4)}")
print(f"Score variance: {bb2_score.var():.6f}")

q70 = bb2_score.quantile(0.70)
q30 = bb2_score.quantile(0.30)
df_bb2['priority_label'] = np.where(bb2_score >= q70, 'High',
                            np.where(bb2_score >= q30, 'Medium', 'Low'))
df_bb2['priority_score'] = bb2_score.round(4)
df_bb2 = df_bb2[kats_cols + ['priority_score']]

print(f"\nBitBrains KATS shape: {df_bb2.shape}")
print(df_bb2['priority_label'].value_counts())

In [ ]:
# Check what actual values look like in Alibaba task table
print("Alibaba task table - key columns stats:")
print(df_alibaba_task[['plan_cpu','plan_mem','plan_gpu','start_time','end_time']].describe())
print("\nGPU type distribution from spec:")
print(df_alibaba_spec['gpu_type'].value_counts())
print("\nStatus distribution:")
print(df_alibaba_task['status'].value_counts().head())

In [ ]:
df_alib_task = df_alibaba_task.copy()

# Compute task duration in seconds
df_alib_task['duration_s'] = (pd.to_numeric(df_alib_task['end_time'], errors='coerce') -
                               pd.to_numeric(df_alib_task['start_time'], errors='coerce')).clip(1, None)

df_al2 = pd.DataFrame()

# service_criticality: GPU allocation drives criticality for AI inference
# plan_gpu 0 → batch/low, plan_gpu > 0 → AI inference/high
plan_gpu = pd.to_numeric(df_alib_task['plan_gpu'], errors='coerce').fillna(0)
plan_cpu = pd.to_numeric(df_alib_task['plan_cpu'], errors='coerce').fillna(1).clip(1, None)
plan_mem = pd.to_numeric(df_alib_task['plan_mem'], errors='coerce').fillna(1).clip(1, None)

# Normalize each to [0,1] using their own distribution
gpu_norm = (plan_gpu / plan_gpu.clip(0.001).quantile(0.99).clip(0.001)).clip(0, 1)
cpu_norm = (plan_cpu / plan_cpu.quantile(0.99)).clip(0, 1)
mem_norm = (plan_mem / plan_mem.quantile(0.99)).clip(0, 1)

df_al2['service_criticality'] = ((gpu_norm * 5 + cpu_norm * 3 + mem_norm * 2) * 9 / 10 + 1).clip(1, 10).round().astype(int)

# RTO from task duration: shorter tasks = tighter RTO tolerance
duration_min = (df_alib_task['duration_s'] / 60).clip(1, 1440)
df_al2['rto_minutes']     = duration_min.round(1)
df_al2['rpo_minutes']     = (duration_min * 0.4).round(1)

# Bandwidth from CPU + memory plan
df_al2['bandwidth_required_mbps'] = (cpu_norm * 500 + mem_norm * 200).round(2)
df_al2['data_volume_gb']           = mem_norm.round(4)
df_al2['active_sessions']          = (gpu_norm * 5000 + cpu_norm * 1000).astype(int)

# downstream_critical: GPU jobs with high CPU = critical inference service
df_al2['downstream_critical']  = ((plan_gpu > 0) & (plan_cpu > plan_cpu.median())).astype(int)
df_al2['migration_complexity'] = pd.cut(plan_cpu, bins=[0,2,4,8,16,9999],
                                          labels=[1,2,3,4,5]).astype(float).fillna(2).astype(int)
df_al2['regulatory_flag']      = 0
df_al2['latency_sensitivity']  = (plan_gpu > 0).astype(int)  # GPU = real-time inference
df_al2['redundancy_level']     = np.random.randint(0, 4, size=len(df_alib_task))
df_al2['az_risk_score']        = (gpu_norm * 0.6 + np.random.uniform(0, 0.4, len(df_alib_task))).clip(0,1).round(4)
df_al2['multiregion_deployed'] = (df_al2['redundancy_level'] >= 2).astype(int)
df_al2['dependency_count']     = np.random.poisson(2, size=len(df_alib_task))
df_al2['service_sector']       = 'ai_inference_alibaba'

al2_score = (0.30 * df_al2['service_criticality'] / 10 +
             0.20 * (1 - df_al2['rto_minutes'].clip(0,1440) / 1440) +
             0.15 * df_al2['regulatory_flag'] +
             0.15 * df_al2['downstream_critical'] +
             0.10 * df_al2['az_risk_score'] +
             0.05 * df_al2['latency_sensitivity'] +
             0.05 * (1 - df_al2['redundancy_level'] / 3))

print(f"Alibaba score stats:\n{al2_score.describe().round(4)}")
print(f"Score variance: {al2_score.var():.6f}")

q70 = al2_score.quantile(0.70)
q30 = al2_score.quantile(0.30)
df_al2['priority_label'] = np.where(al2_score >= q70, 'High',
                            np.where(al2_score >= q30, 'Medium', 'Low'))
df_al2['priority_score'] = al2_score.round(4)
df_al2 = df_al2[kats_cols + ['priority_score']].reset_index(drop=True)

# Sample 50k
df_al2_sample = df_al2.sample(n=50000, random_state=42).reset_index(drop=True)

print(f"\nAlibaba full: {df_al2.shape}")
print(df_al2['priority_label'].value_counts())
print(f"\nAlibaba sample (50k):")
print(df_al2_sample['priority_label'].value_counts())

In [ ]:
df_bb2.to_parquet('/kaggle/working/kats_data/kats_bitbrains.parquet',      index=False)
df_al2_sample.to_parquet('/kaggle/working/kats_data/kats_alibaba.parquet', index=False)

print("✅ All datasets saved and verified!")
print("\n" + "="*65)
print(f"{'Dataset':<26} {'Rows':>8} {'High%':>7} {'Med%':>7} {'Low%':>7}")
print("="*65)
for name, df in [("KATS-SYN (synthetic)",    df_kats_syn),
                  ("Google Borg",             df_borg_kats),
                  ("BitBrains (fixed)",       df_bb2),
                  ("Alibaba GPU (50k)",       df_al2_sample)]:
    vc = df['priority_label'].value_counts(normalize=True) * 100
    print(f"{name:<26} {len(df):>8,}  {vc.get('High',0):>5.1f}%  {vc.get('Medium',0):>5.1f}%  {vc.get('Low',0):>5.1f}%")

In [ ]:
df_bb2 = pd.DataFrame()

mem_util_ratio = (bb_agg2['mem_usage_kb_mean'] / bb_agg2['mem_prov_kb'].clip(1)).clip(0, 1)
cpu_norm       = (bb_agg2['cpu_pct_mean'] / 100).clip(0, 1)
disk_max_val   = float(bb_agg2['disk_write_mean'].max())          # ← scalar first
disk_norm      = (bb_agg2['disk_write_mean'] / max(disk_max_val, 1)).clip(0, 1)

df_bb2['service_criticality'] = ((cpu_norm * 4 + mem_util_ratio * 3 + disk_norm * 3) * 9 + 1).clip(1, 10).round().astype(int)

disk_write_capped             = bb_agg2['disk_write_mean'].clip(0.01, None)
df_bb2['rto_minutes']         = (1000 / disk_write_capped).clip(1, 1440).round(1)
df_bb2['rpo_minutes']         = (df_bb2['rto_minutes'] * 0.5).round(1)
df_bb2['bandwidth_required_mbps'] = ((bb_agg2['net_rx_mean'] + bb_agg2['net_tx_mean']) / 1024 * 8).clip(0.01).round(3)
df_bb2['data_volume_gb']      = (bb_agg2['mem_usage_kb_mean'] / 1024 / 1024).round(3)
df_bb2['active_sessions']     = (bb_agg2['cpu_pct_mean'] * bb_agg2['cpu_cores']).astype(int)
df_bb2['downstream_critical'] = (bb_agg2['cpu_pct_std'].fillna(0) > bb_agg2['cpu_pct_mean'] * 0.5).astype(int)
df_bb2['migration_complexity']= pd.cut(bb_agg2['cpu_cores'],
                                        bins=[0,1,2,4,8,999],
                                        labels=[1,2,3,4,5]).astype(float).fillna(2).astype(int)
df_bb2['regulatory_flag']     = 1
df_bb2['latency_sensitivity'] = 1
df_bb2['redundancy_level']    = np.random.randint(0, 4, size=len(bb_agg2))
df_bb2['az_risk_score']       = (cpu_norm * 0.5 + mem_util_ratio * 0.3 + np.random.uniform(0, 0.2, len(bb_agg2))).clip(0,1).round(4)
df_bb2['multiregion_deployed']= (df_bb2['redundancy_level'] >= 2).astype(int)
df_bb2['dependency_count']    = np.random.poisson(3, size=len(bb_agg2))
df_bb2['service_sector']      = 'financial_bitbrains'

bb2_score = (0.30 * df_bb2['service_criticality'] / 10 +
             0.20 * (1 - df_bb2['rto_minutes'].clip(0,1440) / 1440) +
             0.15 * df_bb2['regulatory_flag'] +
             0.15 * df_bb2['downstream_critical'] +
             0.10 * df_bb2['az_risk_score'] +
             0.05 * df_bb2['latency_sensitivity'] +
             0.05 * (1 - df_bb2['redundancy_level'] / 3))

print(f"BitBrains score variance: {bb2_score.var():.6f}")
print(bb2_score.describe().round(4))

q70 = bb2_score.quantile(0.70)
q30 = bb2_score.quantile(0.30)
df_bb2['priority_label']  = np.where(bb2_score >= q70, 'High',
                             np.where(bb2_score >= q30, 'Medium', 'Low'))
df_bb2['priority_score']  = bb2_score.round(4)
df_bb2 = df_bb2[kats_cols + ['priority_score']]

print(f"\nBitBrains KATS shape: {df_bb2.shape}")
print(df_bb2['priority_label'].value_counts())

In [ ]:
df_bb2.to_parquet('/kaggle/working/kats_data/kats_bitbrains.parquet',      index=False)
df_al2_sample.to_parquet('/kaggle/working/kats_data/kats_alibaba.parquet', index=False)
# KATS-SYN and Borg already saved correctly

print("✅ All 4 KATS-schema datasets saved!\n")
print("=" * 65)
print(f"{'Dataset':<26} {'Rows':>8}  {'High%':>6}  {'Med%':>6}  {'Low%':>6}")
print("=" * 65)
for name, df in [("KATS-SYN (synthetic)",    df_kats_syn),
                  ("Google Borg",             df_borg_kats),
                  ("BitBrains (fixed)",       df_bb2),
                  ("Alibaba GPU (50k)",       df_al2_sample)]:
    vc = df['priority_label'].value_counts(normalize=True) * 100
    print(f"{name:<26} {len(df):>8,}  {vc.get('High',0):>5.1f}%  {vc.get('Medium',0):>5.1f}%  {vc.get('Low',0):>5.1f}%")

print("\nKATS feature schema columns:", kats_cols)
print("\n✅ Ready for Week 5-6: Baseline Models + KATS-Ensemble training")